In [ ]:
import os, re, math, time, json, random, copy, html, urllib.request
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import HTML, display

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(1337)
random.seed(1337)
print("PyTorch:", torch.__version__)
if device == "cuda":
    print("✅ GPU found:", torch.cuda.get_device_name(0), "- training will take ~2 minutes")
else:
    print("⚠️ No GPU. Runtime → Change runtime type → T4 GPU.  (CPU works too: we use a smaller model)")

In [ ]:
RAW_URL = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-valid.txt"
EOT = "<|endoftext|>"
if not os.path.exists("TinyStories-valid.txt"):
    print("Downloading Tinystories (19mb)")
    urllib.request.urlretrieve(RAW_URL, "TinyStories-valid.txt")


In [7]:
raw = open("TinyStories-valid.txt", encoding="utf-8").read()




In [8]:
#........data cleaning :real LLM teams spend a LOT of time here........."

REPLACEMENTS = {"\u2018" : "'","\u2019" : "'","\u201c" : '"',"\u201d" : '"',"\u2013" : "-","\u2014" : "-","\u2026" : "...", "\u00a0" : " "}

In [9]:
def clean(story):
    for bad, good in REPLACEMENTS.items():
        story = story.replace(bad, good)
    return story.strip()


stories = []

for story in raw.split(EOT):
    story = clean(story)
    if len(story) < 50 or not story.isascii():
        continue
    stories.append(story)
    if len(stories) == 6000:
        break

In [ ]:
text = "".join(s + "\n" + EOT + "\n"  for s in stories)
open("stories.txt", "w", encoding="utf-8").write(text)
print(f"Stories : {len(stories):,} Charaters: {len(text):,} words: {len(text.split):,}")


TypeError: object of type 'builtin_function_or_method' has no len()

In [ ]:
print(stories[0])

In [ ]:
words = text.split()
next_words = defaultdict(Counter)

for a,b in zip(words, words[1:]):
    next_words[a][b] += 1

def show_next(word, k=6):
    counts = next_words[word]
    total = sum(counts.values())
    print(f"After '{word}':"," " .join(f"(w) (c / total:.0%)" for w,c in counts.most_common(k)))

show_next("upon")
show_next("Little")
show_next("was")

In [ ]:
def counting_lm(start = "Once", n=30):
    out = [start]
    for _ in range(n):

        options = next_words[out[-1]]
        if not options:
            break
        out.append(random.choice(list(options), weights = list(options.values()))[0])
    return " ".join(out)

print(counting_lm())

In [ ]:
chars = sorted(set(text.replace(EOT, "")))
stoi = {c:i for i, c in enumerate(chars)}

print(len(chars), "characters:", repr("".join(chars)))


